# Explore YOLO training results

Use this notebook to review five views of a saved training run: losses by epoch, validation metrics by epoch, the precision-recall curve, the confusion matrix, and ground-truth-versus-prediction examples. You can optionally run the trained checkpoint on unlabeled real photos.

> **Safety:** Opening or running the analysis cells does not start training. The optional training cell is at the end and only starts if you change `START_TRAINING` to `True`. Real-image prediction is also opt-in.

The saved curves, confusion matrix, and example images are generated from the validation split. The held-out test split is not used here, so these are not final test-set results.

## 1. Select a training run

Run the next cell, then change `RUN_NAME` to another folder listed under `runs/detect` if you want to inspect a different experiment.

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys

import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Resolve the repository even if Jupyter was opened from the notebooks folder.
_working_dir = Path.cwd().resolve()
REPO_ROOT = next((path for path in (_working_dir, *_working_dir.parents) if (path / 'configs').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Could not find the project root. Open this notebook from inside the repository.')
RUNS_ROOT = REPO_ROOT / 'runs' / 'detect'
available_runs = sorted(path.name for path in RUNS_ROOT.iterdir() if path.is_dir()) if RUNS_ROOT.is_dir() else []
print('Available runs:', available_runs if available_runs else 'No runs found under runs/detect')

RUN_NAME = 'palletjack-detection-yolo26n-seed42'  # Change this to another available run folder.
RUN_DIR = RUNS_ROOT / RUN_NAME
print('Selected run:', RUN_DIR)

## 2. Compare losses and validation metrics by epoch

The first chart shows training and validation losses. The second shows precision, recall, mAP50, and mAP50-95 on the validation split. A widening gap between training and validation loss can be a sign of overfitting. If the selected run is incomplete or has no `results.csv`, choose another run and rerun the selection and analysis cells.

In [ ]:
results_csv = RUN_DIR / 'results.csv'
if not RUN_DIR.is_dir():
    print(f'Run folder was not found: {RUN_DIR}. Choose another run from the list above.')
elif not results_csv.is_file():
    print(f'results.csv was not found in {RUN_DIR}. Choose another run or check that training produced results.')
else:
    with results_csv.open(newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))
    if not rows:
        print(f'{results_csv} contains no epoch rows. Choose another run or check the training output.')
    else:
        columns = rows[0].keys()
        epochs = [float(row['epoch']) if row.get('epoch') else index for index, row in enumerate(rows)]
        loss_columns = [column for column in columns if column.endswith('loss')]
        metric_columns = [column for column in columns if column.startswith('metrics/') and column.endswith('(B)')]
        figure, axes = plt.subplots(1, 2, figsize=(15, 5))
        for column in loss_columns:
            values = [float(row[column]) if row.get(column) else float('nan') for row in rows]
            axes[0].plot(epochs, values, label=column)
        axes[0].set(title='Loss by epoch', xlabel='Epoch', ylabel='Loss')
        if loss_columns:
            axes[0].legend()
        for column in metric_columns:
            values = [float(row[column]) if row.get(column) else float('nan') for row in rows]
            axes[1].plot(epochs, values, label=column.removeprefix('metrics/'))
        axes[1].set(title='Validation metrics by epoch', xlabel='Epoch', ylabel='Score')
        axes[1].set_ylim(0, 1)
        if metric_columns:
            axes[1].legend()
        figure.tight_layout()
        plt.show()
        best_row = max(rows, key=lambda row: float(row.get('metrics/mAP50-95(B)', '0') or 0))
        print('Epoch with best validation mAP50-95:', best_row.get('epoch'), best_row.get('metrics/mAP50-95(B)'))

## 3. Inspect the precision-recall curve, confusion matrix, and examples

The precision-recall curve shows the tradeoff as the confidence threshold changes. The confusion matrices summarize validation predictions, and the paired images compare ground-truth boxes with model predictions. These are validation artifacts, not held-out test results.

In [ ]:
for filename in ('BoxPR_curve.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png'):
    artifact = RUN_DIR / filename
    if artifact.is_file():
        title = 'Precision-recall curve' if filename == 'BoxPR_curve.png' else filename
        display(Markdown(f'### {title} (`{filename}`)'))
        display(Image(filename=str(artifact)))
    else:
        print(f'Optional artifact not found: {artifact}')

prediction_images = sorted(RUN_DIR.glob('val_batch*_pred.jpg')) if RUN_DIR.is_dir() else []
if prediction_images:
    for prediction_image in prediction_images[:3]:
        label_image = prediction_image.with_name(prediction_image.name.replace('_pred.jpg', '_labels.jpg'))
        display(Markdown(f'### Validation batch: `{prediction_image.stem}`'))
        if label_image.is_file():
            display(Markdown('Ground truth labels'))
            display(Image(filename=str(label_image)))
        display(Markdown('Model predictions'))
        display(Image(filename=str(prediction_image)))
else:
    print(f'No validation prediction images were found in {RUN_DIR}. Choose another run or check that plots were saved.')

## 4. Optional: inspect predictions on your own real photos

Put photos in `local/real-images`. Set `RUN_REAL_IMAGE_PREDICTION = True` only when you want to run the selected checkpoint on them. The cell displays annotated copies and does not calculate accuracy because these photos are unlabeled.

In [ ]:
RUN_REAL_IMAGE_PREDICTION = False  # Explicit opt-in; change to True to run inference.
REAL_IMAGE_SOURCE = REPO_ROOT / 'local' / 'real-images'
REAL_PREDICTION_OUTPUT = REPO_ROOT / 'runs' / 'predictions' / 'notebook-real-photos'
REAL_IMAGE_CONFIDENCE = 0.25

if RUN_REAL_IMAGE_PREDICTION:
    if not REAL_IMAGE_SOURCE.exists():
        print(f'Real-image folder was not found: {REAL_IMAGE_SOURCE}. Add photos or change REAL_IMAGE_SOURCE.')
    elif not (RUN_DIR / 'weights' / 'best.pt').is_file():
        print(f'best.pt was not found in {RUN_DIR}. Choose a completed run.')
    else:
        sys.path.insert(0, str(REPO_ROOT / 'src'))
        from palletjack_detection.predict_real_images import predict_real_images
        saved_images = predict_real_images(
            source=REAL_IMAGE_SOURCE,
            model_path=RUN_DIR / 'weights' / 'best.pt',
            output_dir=REAL_PREDICTION_OUTPUT,
            confidence=REAL_IMAGE_CONFIDENCE,
            imgsz=640,
        )
        print(f'Saved {len(saved_images)} annotated image(s) to {REAL_PREDICTION_OUTPUT}')
        for saved_image in saved_images:
            display(Image(filename=str(saved_image)))
else:
    print('Real-image inference is off. Set RUN_REAL_IMAGE_PREDICTION = True to run it.')

## 5. Optional: start a training run

> This cell is separate from the analysis cells. It will not train unless you explicitly set `START_TRAINING = True` and run this cell. It uses the saved CLI config and refuses to overwrite an existing run folder.

In [ ]:
START_TRAINING = False  # Explicit opt-in; change to True only when you intend to train.
if START_TRAINING:
    training_script = REPO_ROOT / 'scripts' / 'train_yolo.py'
    training_config = REPO_ROOT / 'configs' / 'train_yolo.yaml'
    subprocess.run(
        [sys.executable, str(training_script), '--config', str(training_config)],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print('Training is off. Set START_TRAINING = True only when you intend to start a run.')